# Tutorial 01 — Language models, tokens, and experimental baselines

[Lecture notes](../lectures/01_tokenization.md) · [Exercise sheet](../exercises/01_tokenization.md)

**40 minutes, CPU, no accounts or model downloads.** Requires Python 3.10+ and NumPy. These are mechanism experiments using authored or synthetic data, not frontier benchmark results. Restart the kernel and run all cells in order.

**Objective:** Compare byte, character, and whitespace tokenization; fit a smoothed character bigram baseline; measure held-out perplexity.

Allocate 5 minutes to setup, 15 to the mechanism, 10 to interventions, and 10 to discussion. Record predictions before running. Complete the separate exercise sheet after class.


In [1]:
import sys
import numpy as np
rng = np.random.default_rng(42)
print("Python", sys.version.split()[0], "NumPy", np.__version__, "seed", 42)


Python 3.13.1 NumPy 2.1.3 seed 42


## 1. Inspect authored examples and tokenization

These short examples are classroom strings, not a language benchmark. Compare units and preserve Unicode. Predict which count changes for a combining accent.

Before running: predict the output or the invariant being checked.


In [2]:
import unicodedata
examples = ["The class starts today.", "Darasi ya fara yau.", "Darasa linaanza leo.", "cafe\u0301", "café"]
for text in examples:
    print(repr(text), {"words": len(text.split()), "codepoints": len(text),
          "bytes": len(text.encode("utf-8")), "NFC_codepoints": len(unicodedata.normalize("NFC", text))})
assert len(examples[-2]) != len(examples[-1])
assert unicodedata.normalize("NFC", examples[-2]) == examples[-1]


'The class starts today.' {'words': 4, 'codepoints': 23, 'bytes': 23, 'NFC_codepoints': 23}
'Darasi ya fara yau.' {'words': 4, 'codepoints': 19, 'bytes': 19, 'NFC_codepoints': 19}
'Darasa linaanza leo.' {'words': 3, 'codepoints': 20, 'bytes': 20, 'NFC_codepoints': 20}
'café' {'words': 1, 'codepoints': 5, 'bytes': 6, 'NFC_codepoints': 4}
'café' {'words': 1, 'codepoints': 4, 'bytes': 5, 'NFC_codepoints': 4}


**Interpretation checkpoint:** Describe the observed result, identify which line implements the key assumption, and propose one change that could invalidate your conclusion. Keep the printed outputs as evidence.


## 2. Fit only on training documents

We estimate a smoothed character bigram distribution. The vocabulary includes an unknown symbol and document boundaries. The held-out document does not fit the vocabulary.

Before running: predict the output or the invariant being checked.


In [3]:
from collections import Counter
train_docs = ["the class starts today", "the lab starts today", "the class ends soon"]
dev_docs = ["the lab ends soon"]
vocab = ["<unk>", "<bos>", "<eos>"] + sorted(set("".join(train_docs)))
ids = {c:i for i,c in enumerate(vocab)}
def encode(doc):
    return [ids["<bos>"]] + [ids.get(c, 0) for c in doc] + [ids["<eos>"]]
counts = np.zeros((len(vocab), len(vocab)))
for doc in train_docs:
    seq = encode(doc)
    for a,b in zip(seq,seq[1:]): counts[a,b] += 1
alpha = 0.5
probs = (counts + alpha) / (counts.sum(axis=1,keepdims=True) + alpha*len(vocab))
assert np.allclose(probs.sum(axis=1),1)
def nll(doc):
    seq=encode(doc)
    return -sum(np.log(probs[a,b]) for a,b in zip(seq,seq[1:])),len(seq)-1
loss,n = nll(dev_docs[0])
print({"dev_nll":loss,"predicted_tokens":n,"dev_perplexity":np.exp(loss/n),"uniform_perplexity":len(vocab)})


{'dev_nll': np.float64(33.17008795105448), 'predicted_tokens': 18, 'dev_perplexity': np.float64(6.314083811403138), 'uniform_perplexity': 17}


**Interpretation checkpoint:** Describe the observed result, identify which line implements the key assumption, and propose one change that could invalidate your conclusion. Keep the printed outputs as evidence.


## 3. Intervene on the smoothing strength

Sweep smoothing on development data. Explain why unseen transitions punish an unsmoothed model. This is development selection, not a final test result.

Before running: predict the output or the invariant being checked.


In [4]:
for alpha in [0.01,0.1,0.5,1.0]:
    probs=(counts+alpha)/(counts.sum(axis=1,keepdims=True)+alpha*len(vocab))
    loss,n=nll(dev_docs[0])
    print("alpha",alpha,"dev PPL",round(float(np.exp(loss/n)),3))
assert np.isclose(np.exp(-np.mean(np.log([.5,.25,.25]))),3.174802103936399)


alpha 0.01 dev PPL 2.901
alpha 0.1 dev PPL 3.764
alpha 0.5 dev PPL 6.314
alpha 1.0 dev PPL 8.261


**Interpretation checkpoint:** Describe the observed result, identify which line implements the key assumption, and propose one change that could invalidate your conclusion. Keep the printed outputs as evidence.


## Practical discussion

1. A tokenizer uses twice as many tokens for Hausa as English on your sample. What can and cannot be concluded?
2. Why is fitting the vocabulary before splitting a form of leakage?
3. How should two sentences from the same source document be split?


## Extension and submission

Use the real tokenizer in `extensions/hf_text.py` to audit the same examples. Record the checkpoint revision; model inference is optional.

See the [extension guide](../extensions/README.md) before running model downloads. Extensions require additional dependencies and are not necessary for full exercise credit.

Submit the completed notebook with a 150–250-word interpretation and the separate exercise answers. Include a baseline, one controlled change, data provenance, seed, and an honest limitation. Never replace measured results with expected trends.
